# Машинный перевод на основе многоязычных эмбеддингов

В этом проекте я создаю систему машинного перевода без использования параллельных корпусов или сложных нейросетевых архитектур.
Модель основывается на идее, что близкородственные языки имеют схожие контекстные распределения слов, и это можно использовать для переноса смыслов между ними.

Для эксперимента выбраны два близких славянских языка — украинский и русский.

Пример различий
(синій кіт vs. синій кит)


##Сходство базовой лексики славянских языков
| Russian         | Belorussian              | Ukrainian               | Polish             | Czech                         | Bulgarian            |
|-----------------|--------------------------|-------------------------|--------------------|-------------------------------|-----------------------|
| женщина         | жанчына, кабета, баба    | жінка                   | kobieta            | žena                          | жена                  |
| мужчина         | мужчына                  | чоловік, мужчина        | mężczyzna          | muž                           | мъж                   |
| человек         | чалавек                  | людина, чоловік         | człowiek           | člověk                        | човек                 |
| ребёнок, дитя   | дзіця, дзіцёнак, немаўля | дитина, дитя            | dziecko            | dítě                          | дете                  |
| жена            | жонка                    | дружина, жінка          | żona               | žena, manželka, choť          | съпруга, жена         |
| муж             | муж, гаспадар            | чоловiк, муж            | mąż                | muž, manžel, choť             | съпруг, мъж           |
| мать, мама      | маці, матка              | мати, матір, неня, мама | matka              | matka, máma, 'стар.' mateř    | майка                 |
| отец, тятя      | бацька, тата             | батько, тато, татусь    | ojciec             | otec                          | баща, татко           |
| много           | шмат, багата             | багато                  | wiele              | mnoho, hodně                  | много                 |
| несколько       | некалькі, колькі         | декілька, кілька        | kilka              | několik, pár, trocha          | няколко               |
| другой, иной    | іншы                     | інший                   | inny               | druhý, jiný                   | друг                  |
| зверь, животное | жывёла, звер, істота     | тварина, звір           | zwierzę            | zvíře                         | животно               |
| рыба            | рыба                     | риба                    | ryba               | ryba                          | риба                  |
| птица           | птушка                   | птах, птиця             | ptak               | pták                          | птица                 |
| собака, пёс     | сабака                   | собака, пес             | pies               | pes                           | куче, пес             |
| вошь            | вош                      | воша                    | wesz               | veš                           | въшка                 |
| змея, гад       | змяя                     | змія, гад               | wąż                | had                           | змия                  |
| червь, червяк   | чарвяк                   | хробак, черв'як         | robak              | červ                          | червей                |
| дерево          | дрэва                    | дерево                  | drzewo             | strom, dřevo                  | дърво                 |
| лес             | лес                      | ліс                     | las                | les                           | гора, лес             |
| палка           | кій, палка               | палиця                  | patyk, pręt, pałka | hůl, klacek, prut, kůl, pálka | палка, пръчка, бастун |



Эти примеры показывают, что языки схожи не только по форме слов, но и по контекстам, в которых они употребляются. Это свойство можно использовать при построении модели перевода.

In [1]:
!pip install gensim

Загружаю предобученные эмбеддинги для украинского и русского языков:
* [cc.uk.300.vec.zip](https://yadi.sk/d/9CAeNsJiInoyUA)
* [cc.ru.300.vec.zip](https://yadi.sk/d/3yG0-M4M8fypeQ)

In [10]:
import gensim
import numpy as np
from gensim.models import KeyedVectors

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [4]:
!unzip "/content/drive/MyDrive/Colab Notebooks/DL/NLP/shad_yandex_data_scholl_2023/week1/cc.uk.300.vec.zip" -d "/content/"
!unzip "/content/drive/MyDrive/Colab Notebooks/DL/NLP/shad_yandex_data_scholl_2023/week1/cc.ru.300.vec.zip" -d "/content/"

!ls /content | grep cc.uk
!ls /content | grep cc.uk


Archive:  /content/drive/MyDrive/Colab Notebooks/DL/NLP/shad_yandex_data_scholl_2023/week1/cc.uk.300.vec.zip
  inflating: /content/cc.uk.300.vec  
Archive:  /content/drive/MyDrive/Colab Notebooks/DL/NLP/shad_yandex_data_scholl_2023/week1/cc.ru.300.vec.zip
  inflating: /content/cc.ru.300.vec  
cc.uk.300.vec
cc.uk.300.vec


In [6]:
path_uk_emb = "/content/cc.uk.300.vec"
path_ru_emb = "/content/cc.ru.300.vec"
uk_emb = KeyedVectors.load_word2vec_format(path_uk_emb)
ru_emb = KeyedVectors.load_word2vec_format(path_ru_emb)

In [8]:
ru_emb.most_similar([ru_emb["август"]], topn=10)
uk_emb.most_similar([uk_emb["серпень"]])
ru_emb.most_similar([uk_emb["серпень"]])

[('Недопустимость', 0.24435284733772278),
 ('конструктивность', 0.23293082416057587),
 ('офор', 0.23256804049015045),
 ('deteydlya', 0.230317160487175),
 ('пресечении', 0.22632381319999695),
 ('одностороннего', 0.22608886659145355),
 ('подход', 0.2230587750673294),
 ('иболее', 0.22003726661205292),
 ('2015Александр', 0.21872766315937042),
 ('конструктивен', 0.21796567738056183)]

Загружаю небольшие словари пар слов, которые используются как обучающая и тестовая выборки:

In [12]:
def load_word_pairs(filename):
    uk_ru_pairs = []
    uk_vectors = []
    ru_vectors = []
    with open(filename, "r") as inpf:
        for line in inpf:
            uk, ru = line.rstrip().split("\t")
            if uk not in uk_emb or ru not in ru_emb:
                continue
            uk_ru_pairs.append((uk, ru))
            uk_vectors.append(uk_emb[uk])
            ru_vectors.append(ru_emb[ru])
    return uk_ru_pairs, np.array(uk_vectors), np.array(ru_vectors)

[Отсланые файлы нужно скачать с GitHub](https://github.com/yandexdataschool/nlp_course/tree/2023/week01_embeddings)

In [14]:
uk_ru_train, X_train, Y_train = load_word_pairs("/content/drive/MyDrive/Colab Notebooks/DL/NLP/shad_yandex_data_scholl_2023/week1/ukr_rus.train.txt")
uk_ru_test, X_test, Y_test = load_word_pairs("/content/drive/MyDrive/Colab Notebooks/DL/NLP/shad_yandex_data_scholl_2023/week1/ukr_rus.test.txt")


## Отображение векторных пространств

Пусть $x_i \in \mathrm{R}^d$ — распределённое представление слова \(i\) в исходном языке,  
а $y_i \in \mathrm{R}^d$ — векторное представление его перевода.  

Наша цель — найти такое линейное преобразование $W$, которое минимизирует евклидово расстояние между $Wx_i$ и $y_i$ для некоторого подмножества словарных векторов.  
Эта задача известна как **проблема Прокруста**:

$$W^* = \arg\min_W \sum_{i=1}^n||Wx_i - y_i||_2$$  
или  
$$W^* = \arg\min_W ||WX - Y||_F$$  

где $||*||_F$ — норма Фробениуса.  

В древнегреческой мифологии Прокруст (от греч. «растягивающий») был разбойником из Аттики,  
который заставлял путников подгонять своё тело под размер железной кровати — растягивая или обрезая ноги.  
Мы делаем нечто похожее с исходным пространством эмбеддингов:  
наше «прокрустово ложе» — это целевое пространство векторов.


![embedding_mapping.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/embedding_mapping.png)

![procrustes.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/procrustes.png)

##Линейное преобразование эмбеддингов

Для нахождения соответствия между украинскими и русскими векторными пространствами я обучаю простое линейное отображение без свободного члена:

In [15]:
from sklearn.linear_model import LinearRegression

mapping = LinearRegression(fit_intercept=False)
mapping.fit(X_train, Y_train)


LinearRegression(fit_intercept=False)

После обучения посмотрим, как изменился вектор украинского слова "серпень" (аналог русского "август") после линейного преобразования:

In [16]:
august = mapping.predict(uk_emb["серпень"].reshape(1, -1))
ru_emb.most_similar(august)


[('апрель', 0.8541286587715149),
 ('июнь', 0.8411202430725098),
 ('март', 0.839699387550354),
 ('сентябрь', 0.8359869718551636),
 ('февраль', 0.832929790019989),
 ('октябрь', 0.8311845660209656),
 ('ноябрь', 0.8278924226760864),
 ('июль', 0.823452889919281),
 ('август', 0.8120501637458801),
 ('декабрь', 0.803900420665741)]

После трансформации среди ближайших соседей действительно появляются названия месяцев, однако правильный вариант находится примерно на девятом месте.`

## Метрика качества

Для оценки точности перевода я использую precision@k, где:

top-1 — правильный перевод находится на первом месте;

top-5 — среди пяти ближайших соседей;

top-10 — среди десяти ближайших соседей.

In [17]:
def precision(pairs, mapped_vectors, topn=1):
    """
    pairs — список правильных пар [(uk_word, ru_word), ...]
    mapped_vectors — вектора после линейного отображения
    topn — число ближайших соседей, среди которых ищем правильный перевод
    """
    assert len(pairs) == len(mapped_vectors)
    num_matches = 0
    for i, (_, ru) in enumerate(pairs):
        mapped_vector = mapped_vectors[i]
        most_similar_words = ru_emb.most_similar([mapped_vector], topn=topn)
        top_words = [word for word, _ in most_similar_words]
        if ru in top_words:
            num_matches += 1
    precision_val = num_matches / len(pairs)
    return precision_val


Проверка корректности работы функции:

In [18]:
assert precision([("серпень", "август")], august, topn=5) == 0.0
assert precision([("серпень", "август")], august, topn=9) == 1.0
assert precision([("серпень", "август")], august, topn=10) == 1.0

assert precision(uk_ru_test, X_test) == 0.0
assert precision(uk_ru_test, Y_test) == 1.0


Вычисление точности модели на тестовых данных:

In [19]:
precision_top1 = precision(uk_ru_test, mapping.predict(X_test), 1)
precision_top5 = precision(uk_ru_test, mapping.predict(X_test), 5)

assert precision_top1 >= 0.635
assert precision_top5 >= 0.813

print(precision_top1)
print(precision_top5)


0.6356589147286822
0.813953488372093


## Улучшаем качество (ортогональная задача Прокруста)

Можно показать (см. оригинальную статью), что самосогласованное линейное отображение между семантическими пространствами должно быть **ортогональным**.  
Мы можем наложить на матрицу преобразования $W$ ограничение ортогональности. Тогда получаем следующую задачу:

$$W^* = \arg\min_W ||WX - Y||_F, \text{ где } W^T W = I$$

$$I \text{ — единичная матрица}$$

Вместо того чтобы снова решать задачу линейной регрессии, можно найти оптимальное ортогональное преобразование с помощью **сингулярного разложения (SVD)**.  
Оказывается, что оптимальное преобразование $W^*$ выражается через компоненты SVD следующим образом:

$$X^T Y = U \Sigma V^T \text{ — сингулярное разложение}$$  
$$W^* = U V^T$$


In [21]:
def learn_transform(X_train, Y_train):
    """
    :returns: W* : матрица float [emb_dim x emb_dim], как определено в формулах выше
    """
    # Шаг 1: Вычисляем матрицу ковариации C = X^T * Y
    # Оператор @ в numpy означает матричное умножение.
    C = np.dot(X_train.T, Y_train)

    # Шаг 2: Применяем сингулярное разложение (SVD) к матрице C.
    # np.linalg.svd возвращает три матрицы: U, S (вектор сингулярных значений) и V^T.
    # Нам не нужна матрица масштабирования S для вычисления W.
    U, _, Vt = np.linalg.svd(C)

    # Шаг 3: Вычисляем оптимальную ортогональную матрицу W* = U * V^T
    W_star = np.dot(U, Vt)

    return W_star


Обучаем новую, ортогональную матрицу преобразования

In [22]:
W = learn_transform(X_train, Y_train)

# Проверяем, как теперь переводятся соседи для слова "серпень"
# np.matmul(uk_emb["серпень"], W) — это W*x, где x — вектор слова "серпень"
ru_emb.most_similar([np.matmul(uk_emb["серпень"], W)])


# Проверим качество с новой матрицей W
print(precision(uk_ru_test, np.matmul(X_test, W)))
print(precision(uk_ru_test, np.matmul(X_test, W), 5))

assert precision(uk_ru_test, np.matmul(X_test, W)) >= 0.653
assert precision(uk_ru_test, np.matmul(X_test, W), 5) >= 0.824

0.6537467700258398
0.8242894056847545


## Переводчик UK → RU

In [24]:
# Этот файл также скачиваем с GitHub
path_fairy_tale = "/content/drive/MyDrive/Colab Notebooks/DL/NLP/shad_yandex_data_scholl_2023/week1/fairy_tale.txt"
with open(path_fairy_tale, "r") as inpf:
    uk_sentences = [line.rstrip().lower() for line in inpf]


In [25]:
# Теперь можно реализовать простой переводчик на уровне слов:
# для каждого слова в украинском тексте ищем ближайшее соответствие в русском пространстве.

def translate(sentence):
    """
    :args:
        sentence — предложение на украинском (str)
    :returns:
        translation — предложение на русском (str)

    * ищем вектор для каждого слова в украинской модели
    * преобразуем украинский вектор в русское пространство
    * находим ближайшее русское слово и заменяем им
    """
    tokens = sentence.split(' ')
    translated_tokens = []

    for token in tokens:
        if token in uk_emb:
            uk_vector = uk_emb[token]
            ru_vector = np.matmul(uk_vector, W)
            translated_word = ru_emb.most_similar([ru_vector], topn=1)[0][0]
            translated_tokens.append(translated_word)
        else:
            translated_tokens.append(token)

    return ' '.join(translated_tokens)



In [26]:
# Проверим корректность
assert translate(".") == "."
assert translate("1 , 3") == "1 , 3"
assert translate("кіт зловив мишу") == "кот поймал мышку"

# Переведем весь текст из файла
for sentence in uk_sentences:
    print("src: {}\ndst: {}\n".format(sentence, translate(sentence)))



src: лисичка - сестричка і вовк - панібрат
dst: лисичка – сестричка и волк – панібрат

src: як була собі лисичка та зробила хатку, та й живе. а це приходять холоди. от лисичка замерзла та й побігла в село вогню добувать, щоб витопити. прибігає до одної баби та й каже:
dst: как была себе лисичка и сделала хатку, и и живе. а оно приходят холоди. из лисичка замерзла и и побежала во село огня добувать, чтобы витопити. прибегает к одной бабы и и каже:

src: — здорові були, бабусю! з неділею... позичте мені огню, я вам одслужу.
dst: — здоровые були, бабусю! со неділею... позичте мне огню, мной тебе одслужу.

src: — добре, — каже, — лисичко - сестричко. сідай погрійся трохи, поки я пиріжечки повибираю з печі!
dst: — добре, — каже, — лисичко – сестричко. садись погрійся трохи, пока мной пирожки повибираю со печі!

src: а баба макові пиріжки пекла. от баба вибирає пиріжки та на столі кладе, щоб прохололи; а лисичка підгляділа та за пиріг, та з хати... виїла мачок із середини, а туди напхала смі

Неплохой результат!
Точность можно улучшить, если добавить языковую модель
и использовать несколько ближайших соседей в общем пространстве эмбеддингов.
Но это уже задача для следующего раза.